In [ ]:
import os
import random
import numpy as np
import pandas as pd

import cv2
from PIL import Image
import albumentations as A

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import torch
from tqdm.notebook import tqdm
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import transforms as T
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
set_seed(0)

In [ ]:
def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            name.append(filename.split('.')[0])
    return pd.DataFrame({'id': name}, index = np.arange(0, len(name)))

In [ ]:
def rgb_to_class_mask(rgb_mask, color_to_class):
    class_mask = np.zeros((rgb_mask.shape[0], rgb_mask.shape[1]), dtype=np.int64)
    for color, class_index in color_to_class.items():
        class_mask[np.all(rgb_mask == color, axis=-1)] = class_index
    return class_mask

In [ ]:
def pixel_accuracy(output, mask):
    with torch.no_grad():
        output = torch.argmax(F.softmax(output, dim=1), dim=1)
        correct = torch.eq(output, mask).int()
        accuracy = float(correct.sum()) / float(correct.numel())
    return accuracy

In [ ]:
def miou(pred_mask, mask, smooth=1e-10, n_classes=8):
    with torch.no_grad():
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)
        pred_mask = pred_mask.contiguous().view(-1)
        mask = mask.contiguous().view(-1)

        iou_per_class = []
        for clas in range(0, n_classes):
            true_class = pred_mask == clas
            true_label = mask == clas

            if true_label.long().sum().item() == 0: # no exist label in this loop
                iou_per_class.append(np.nan)
            else:
                intersect = torch.logical_and(true_class, true_label).sum().float().item()
                union = torch.logical_or(true_class, true_label).sum().float().item()
                iou = (intersect + smooth) / (union + smooth)
                iou_per_class.append(iou)
                
        valid_iou = [iou for iou in iou_per_class if not np.isnan(iou)]
        return np.mean(valid_iou)

In [ ]:
def wiou(pred_mask, mask, smooth=1e-10, n_classes=8):
    with torch.no_grad():
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)

        if mask.dim() == 4:
            mask = mask.squeeze(1)

        batch_size = pred_mask.size(0)
        batch_ious = []
        for b in range(batch_size):
            pred_flat = pred_mask[b].contiguous().view(-1)
            mask_flat = mask[b].contiguous().view(-1)
            iou_per_class = []
            total_true_label_count = 0
            for clas in range(n_classes):
                true_class = pred_flat == clas
                true_label = mask_flat == clas
                true_label_count = true_label.long().sum().item()
                total_true_label_count += true_label_count

                if true_label_count == 0: # no exist label in this loop
                    iou_per_class.append(np.nan)
                else:
                    intersect = torch.logical_and(true_class, true_label).sum().float().item()
                    union = torch.logical_or(true_class, true_label).sum().float().item()
                    # add a smoothing term to avoid division by zero
                    iou = (intersect + smooth) / (union + smooth)
                    weight_iou = true_label_count * iou
                    iou_per_class.append(weight_iou)

            valid_iou = [iou for iou in iou_per_class if not np.isnan(iou)]
            img_iou = np.sum(valid_iou) / total_true_label_count if total_true_label_count > 0 else 0.0
            batch_ious.append(img_iou)
        return np.mean(batch_ious)

In [ ]:
class CloudTestDataset(Dataset):
    
    def __init__(self, img_path, mask_path, X, transform=None):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.transform = transform
        self.color_to_class = {
            (  0,   0,   0): 0, # Clear
            (255,  75,  75): 1, # Cirrus
            (255, 150,   0): 2, # Cirrostratus
            (255, 210,  75): 3, # Stratus
            (175, 225, 130): 4, # Stratocumulus
            (130, 175, 225): 5, # Cumulus
            (175, 150, 255): 6, # Cirrocumulus
            (150, 150, 150): 7, # Nimbus
        }
      
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = np.array(cv2.imread(self.mask_path + self.X[idx] + '_mask.png', cv2.IMREAD_COLOR))
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = rgb_to_class_mask(mask, self.color_to_class)
        
        if self.transform is not None:
            aug = self.transform(image=img, mask=mask)
            img = Image.fromarray(aug['image'])
            mask = aug['mask']
        
        if self.transform is None:
            img = Image.fromarray(img)
        
        mask = torch.from_numpy(mask).long()
        
        return img, mask

In [ ]:
batch_size = 4
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [ ]:
model = smp.Unet('timm-mobilenetv3_large_100', encoder_weights='imagenet', classes=8,
                 activation=None, encoder_depth=5, decoder_channels=[256, 128, 64, 32, 16])
model.load_state_dict(torch.load('./Unet-wiou-0.721_loss-0.777_state_dict.pt', weights_only=False))

In [ ]:
IMAGE_PATH_VAL = f'./Data/valid/images/'
MASK_PATH_VAL = f'./Data/valid/masks/'
df_val = create_df(IMAGE_PATH_VAL)
X_val = df_val['id'].values

In [ ]:
t_test = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA)])
test_set = CloudTestDataset(IMAGE_PATH_VAL, MASK_PATH_VAL, X_val, transform=t_test)

### All Class Metrics

In [ ]:
def predict_image_mask_pixel(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image, mask = image.to(device), mask.to(device)
    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        acc = pixel_accuracy(output, mask)
        masked = torch.argmax(output, dim=1)
        masked = masked.cpu().squeeze(0)
    return masked, acc

def pixel_acc(model, test_set):
    accuracy = []
    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        pred_mask, acc = predict_image_mask_pixel(model, img, mask)
        accuracy.append(acc)
    return accuracy

In [ ]:
mob_acc = pixel_acc(model, test_set)

In [ ]:
print('Test Set Pixel Accuracy', np.mean(mob_acc))

In [ ]:
def predict_image_mask_miou(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image, mask = image.to(device), mask.to(device)
    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        score = miou(output, mask)
        masked = torch.argmax(output, dim=1)
        masked = masked.cpu().squeeze(0)
    return masked, score

def miou_score(model, test_set):
    score_iou = []
    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        pred_mask, score = predict_image_mask_miou(model, img, mask)
        score_iou.append(score)
    return score_iou

In [ ]:
mob_miou = miou_score(model, test_set)

In [ ]:
print('Test Set miou', np.mean(mob_miou))

In [ ]:
def predict_image_mask_wiou(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image, mask = image.to(device), mask.to(device)
    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        score = wiou(output, mask)
        masked = torch.argmax(output, dim=1)
        masked = masked.cpu().squeeze(0)
    return masked, score

def wiou_score(model, test_set):
    score_iou = []
    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        pred_mask, score = predict_image_mask_wiou(model, img, mask)
        score_iou.append(score)
    return score_iou

In [ ]:
mob_wiou = wiou_score(model, test_set)

In [ ]:
print('Test Set wiou', np.mean(mob_wiou))

### Per Class Metrics

In [ ]:
def class_predict_image_mask_pixel(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image, mask = image.to(device), mask.to(device)
    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        pred_mask = torch.argmax(output, dim=1).squeeze(0)
        pred_mask = pred_mask.cpu()
        mask = mask.squeeze(0).cpu()
    return pred_mask, mask

def count_pixels_per_class(mask, n_classes=8):
    counts = [0] * n_classes
    for c in range(n_classes):
        counts[c] = int((mask == c).sum().item())
    return counts

def dataset_pixel_counts(model, test_set, n_classes=8):
    model.eval()
    confusion_matrix = np.zeros((n_classes, n_classes), dtype=np.int64)

    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        pred_mask, true_mask = class_predict_image_mask_pixel(model, img, mask)

        for true_class in range(n_classes):
            for pred_class in range(n_classes):
                confusion_matrix[true_class, pred_class] += ((true_mask == true_class) & (pred_mask == pred_class)).sum().item()

    total_true = confusion_matrix.sum(axis=1)
    total_correct_per_class = np.diag(confusion_matrix)

    # Calculate per-class accuracy, handling division by zero
    per_class_accuracy = [
        (correct / total if total > 0 else 0.0)
        for correct, total in zip(total_correct_per_class, total_true)
    ]

    return total_true, per_class_accuracy, confusion_matrix

In [ ]:
# 0, # Clear
# 1, # Cirrus
# 2, # Cirrostratus
# 3, # Stratus
# 4, # Stratocumulus
# 5, # Cumulus
# 6, # Cirrocumulus
# 7, # Nimbus

In [ ]:
true_pixels, acc_per_class, confusion_matrix = dataset_pixel_counts(model, test_set, n_classes=8)
for i in range(8):
    print(f"Class {i} — True pixels: {true_pixels[i]/np.sum(true_pixels)*100:5.2f}% | Accuracy: {acc_per_class[i]:.4f}")

In [ ]:
class_labels = [
    "Clear", "Cirrus", "Cirrostratus", "Stratus",
    "Stratocumulus", "Cumulus", "Cirrocumulus", "Nimbus"
]

# Calculate the percentage of misclassifications
misclassification_perc = np.zeros_like(confusion_matrix, dtype=float)
for i in range(8):
    true_class_total = np.sum(confusion_matrix[i, :])
    if true_class_total > 0:
        misclassification_perc[i, :] = (confusion_matrix[i, :] / true_class_total) * 100

# Create a DataFrame for better visualization
df_misclassification = pd.DataFrame(misclassification_perc, 
                                    index=pd.Index(class_labels, name='True Class'), 
                                    columns=pd.Index(class_labels, name='Predicted Class'))

df_misclassification.style.format('{:.1f}%')

### Figure Show

In [ ]:
# discrete_cmap = ListedColormap([
#     [  0/255,   0/255,   0/255], # Clear: Black
#     [255/255,  75/255,  75/255], # Cirrus: Light Red
#     [255/255, 150/255,   0/255], # Cirrostratus: Light Orange
#     [255/255, 210/255,  75/255], # Stratus: Light Yellow
#     [175/255, 225/255, 130/255], # Stratocumulus: Light Green
#     [130/255, 175/255, 225/255], # Cumulus: Light Blue
#     [175/255, 150/255, 255/255], # Cirrocumulus: Light Purple
#     [150/255, 150/255, 150/255]  # Nimbus: Gray
# ])

In [ ]:
# for i in range(0, len(test_set)):
#     image1, mask1 = test_set[i]
#     fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(20,10))
#     pred_mask1, score1 = predict_image_mask_wiou(model, image1, mask1)
#     ax1.imshow(image1)
#     ax1.set_title('Picture')

#     ax2.imshow(mask1, cmap=discrete_cmap, vmin=0, vmax=7)
#     ax2.set_title('Ground truth')
#     ax2.set_axis_off()

#     ax3.imshow(pred_mask1, cmap=discrete_cmap, vmin=0, vmax=7)
#     ax3.set_title('UNet-MobileNet | wIoU {:.3f}'.format(score1))
#     ax3.set_axis_off()

In [ ]:
# for i in range(80, len(test_set)):
#     image1, mask1 = test_set[i]
#     fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 10), gridspec_kw={'wspace': 0.1})
#     pred_mask1, score1 = predict_image_mask_wiou(model, image1, mask1)
#     ax1.imshow(image1)
#     ax1.set_title('Image', fontsize=21)
#     ax1.set_axis_off()

#     ax2.imshow(mask1, cmap=discrete_cmap, vmin=0, vmax=7)
#     ax2.set_title('Human Label', fontsize=21)
#     ax2.set_axis_off()

#     ax3.imshow(pred_mask1, cmap=discrete_cmap, vmin=0, vmax=7)
#     ax3.set_title('Model Output', fontsize=21)
#     ax3.set_axis_off()